# Receipt item extraction — raw pull

Runs the **real, current extraction pipeline** (`backend/app/services/local_extractor.extract_text` + `backend/app/services/receipt_text_parser.parse_receipt_text_offline` — nothing reimplemented) over every file in `receipts/`, and collects the same data the API normally writes to Supabase's `receipts` + `receipt_items` tables (see `backend/app/api/receipts.py`, `backend/app/db/receipt_items_repo.py`).

**Deliberately out of scope here:** no OpenFoodFacts / BLS matching, no fallback-category-based nutrition estimate (that's `services/fallback_categories.py`, called from the `/receipts/{id}/mapping` endpoint — a separate step). The offline text parser does compute a coarse `category` per item internally (its own static keyword guess, used later as a fallback-matching input) — we deliberately **drop that field** from the collected dataset below so this notebook stays a pure extraction pull.

Nothing here writes to Supabase or any network service — everything runs local (Tesseract OCR + PyMuPDF), same as the app's own receipt pipeline.

In [1]:
import sys
from pathlib import Path

# Make sure `backend` is importable regardless of where Jupyter's cwd ends up.
_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "backend").is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        REPO_ROOT = _candidate
        break
else:
    raise RuntimeError("Could not find the repo root (a parent containing 'backend/') from cwd=" + str(_here))

import mimetypes
from uuid import uuid4

import pandas as pd

from backend.app.services import local_extractor
from backend.app.services.receipt_text_parser import parse_receipt_text_offline

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 300)

RECEIPTS_DIR = REPO_ROOT / "receipts"
REPO_ROOT, RECEIPTS_DIR

(PosixPath('/Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer'),
 PosixPath('/Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipts'))

## Discover receipt files

Every file directly inside `receipts/` (PDFs and phone-photo images — jpg/jpeg/png/etc), sorted for a stable run order.

In [2]:
receipt_files = sorted(p for p in RECEIPTS_DIR.iterdir() if p.is_file() and not p.name.startswith("."))
print(f"{len(receipt_files)} receipt files found")
for p in receipt_files:
    print(" -", p.name)

21 receipt files found
 - 2026.06.30_23001961862026063034796.jpg.png
 - IMG_1928.JPG
 - IMG_1929.JPG
 - IMG_1930.JPG
 - IMG_1931.JPG
 - IMG_1932.JPG
 - IMG_1933.JPG
 - IMG_1934.JPG
 - IMG_1936.JPG
 - IMG_1937.PNG
 - Netto_Kassenbon_20260702-191137.pdf
 - Netto_Kassenbon_20260707-180115.pdf
 - Netto_Kassenbon_20260709-172131.pdf
 - Netto_Kassenbon_20260713-170620.pdf
 - Netto_Kassenbon_20260714-133412.pdf
 - Netto_Kassenbon_20260716-095742.pdf
 - WhatsApp Image 2026-07-05 at 11.02.48.jpeg
 - WhatsApp Image 2026-07-16 at 13.05.02.jpeg
 - WhatsApp Image 2026-07-16 at 13.05.29.jpeg
 - WhatsApp Image 2026-07-16 at 13.05.48.jpeg
 - WhatsApp Image 2026-07-16 at 13.07.58.jpeg


## Extraction

For each file: `local_extractor.extract_text` (PyMuPDF text layer for PDFs, Tesseract OCR for images) -> `parse_receipt_text_offline` (store/date detection + line-item parsing). Mirrors `receipt_parser.scan_receipt_bytes` minus the mock-mode branch, since we want the real extractor to actually run on every file.

One `receipts`-shaped record per file, one `receipt_items`-shaped record per parsed line item (joined by a locally-generated `receipt_id`, same as `upload_receipt` does with `uuid4()`).

In [3]:
receipts_rows = []
items_rows = []

for path in receipt_files:
    receipt_id = str(uuid4())
    file_bytes = path.read_bytes()
    file_type = mimetypes.guess_type(path.name)[0]

    receipt_row = {
        "receipt_id": receipt_id,
        "file_name": path.name,
        "file_type": file_type,
        "status": None,
        "store": None,
        "purchase_date": None,
        "scan_quality": None,
        "items_count": 0,
        "non_food_items_ignored_count": 0,
        "error": None,
    }

    try:
        raw_text = local_extractor.extract_text(file_bytes, path.name)
        parsed = parse_receipt_text_offline(raw_text)

        receipt_row["status"] = "processed"
        receipt_row["store"] = parsed.get("store")
        receipt_row["purchase_date"] = parsed.get("date")
        receipt_row["scan_quality"] = parsed.get("scan_quality")
        receipt_row["items_count"] = parsed.get("items_count", 0)
        receipt_row["non_food_items_ignored_count"] = len(parsed.get("non_food_items_ignored", []) or [])

        for item in parsed.get("items", []) or []:
            items_rows.append({
                "item_id": str(uuid4()),
                "receipt_id": receipt_id,
                "file_name": path.name,

                # Core NLP outputs (receipt_items.raw_name / normalized_name)
                "raw_name": item.get("original_text") or item.get("name"),
                "normalized_name": item.get("name"),

                # Quantitative data
                "quantity": item.get("quantity"),
                "unit": item.get("unit"),
                "price": item.get("price"),

                # NOTE: no "category" here on purpose — that's the fallback-
                # category step (services/fallback_categories.py), out of
                # scope for this raw-extraction pull.
                "matched_product_id": None,  # OFF/BLS matching not run here

                # Confidence signal — identical rule to receipt_items_repo.insert_receipt_items
                "confidence": 0.5 if item.get("uncertain") else 0.9,
                "uncertain": item.get("uncertain"),
            })

    except local_extractor.UnreadableReceipt as e:
        receipt_row["status"] = "error"
        receipt_row["error"] = str(e)

    receipts_rows.append(receipt_row)

print(f"{len(receipts_rows)} receipts processed, {len(items_rows)} items extracted")
print(f"{sum(1 for r in receipts_rows if r['status'] == 'error')} receipt(s) failed to extract")

21 receipts processed, 119 items extracted
0 receipt(s) failed to extract


## Results

`df_receipts` mirrors the `receipts` table (one row per file); `df_items` mirrors `receipt_items` (one row per parsed line item).

In [4]:
df_receipts = pd.DataFrame(receipts_rows)
df_receipts

,receipt_id,file_name,file_type,status,store,purchase_date,scan_quality,items_count,non_food_items_ignored_count,error
0,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,image/png,processed,Lidl,2026-06-30,good,15,3,None
1,8d554238-2f93-4096-9d0c-e0a516bcda25,IMG_1928.JPG,image/jpeg,processed,Netto,2026-06-17,good,2,0,None
2,30bb7241-c4aa-4d61-ba68-9eeffa94c335,IMG_1929.JPG,image/jpeg,processed,Edeka,NaN,good,8,1,None
3,7f34b522-89f4-46ef-a383-ac477898223c,IMG_1930.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
4,88607fdc-8c57-41dd-a442-4a816eb4a99c,IMG_1931.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
5,ccaa60fc-a78b-4a83-8275-30fe96a0772f,IMG_1932.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
6,e2f9af20-46d7-4075-a3c2-627d17dbdba1,IMG_1933.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
7,41101fe2-bb90-43d7-abad-c8853c666c5a,IMG_1934.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
8,1c0e4ee0-1ddd-43b3-a15a-2a0ce3dc0908,IMG_1936.JPG,image/jpeg,processed,unknown,NaN,poor,0,0,None
9,0c84d1cd-e2b1-4873-af7a-31d06e600720,IMG_1937.PNG,image/png,processed,Kaufland,2026-06-15,good,1,0,None


In [5]:
df_items = pd.DataFrame(items_rows)
df_items

,item_id,receipt_id,file_name,raw_name,normalized_name,quantity,unit,price,matched_product_id,confidence,uncertain
0,1827153b-b012-453b-926f-c3df158cae91,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Banane Fair Bio 2,35 A",Banane Fair Bio,1.182,kg,2.35,None,0.5,True
1,592f30db-ea4d-4418-a314-b03d96f7a074,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Zwiebeln weiß 1,28 A",Zwiebeln weiß,0.356,kg,1.28,None,0.5,True
2,97af5108-17a5-480b-af76-e0f0b2e841f3,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bioland Zucchini 1,49 A",Bioland Zucchini,1.000,piece,1.49,None,0.5,True
3,c3138c2f-675b-4c0d-9b6b-37c3472fcd15,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Petersilie 1,39 A",Petersilie,1.000,piece,1.39,None,0.5,True
4,517fc874-ad36-4723-9408-0ba10f7861d5,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Limetten 1,99 A",Bio Limetten,1.000,piece,1.99,None,0.5,True
5,c9574418-4e27-40cd-b21c-4fab8a121b8b,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Weidemilch 3,8% 1,19 A",Weidemilch,1.000,piece,1.19,None,0.5,True
6,5cc0d5e7-9b8b-483e-a5c4-0159e175d6a9,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Exquisa fitlineNatur 1,59 A",Exquisa fitlineNatur,1.000,piece,1.59,None,0.5,True
7,6bf23279-4d79-4a28-a525-71f9f4ee9f2c,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio-Eier OKT 10er 3,99 A",Bio-Eier OKT 10er,1.000,piece,3.99,None,0.5,True
8,1edac537-14b2-4b64-a470-76646701ce9e,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Müsli Schoko Kn. 2,65 A",Bio Müsli Schoko Kn,1.000,piece,2.65,None,0.5,True
9,cd8d9096-6e3c-420a-b96e-c2e89daa45c6,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Lava Bar Cookie 1,69 A",Lava Bar Cookie,1.000,piece,1.69,None,0.5,True


### Combined view

Items joined back with their receipt's metadata (file name, store, purchase date) — one flat table, easiest to eyeball or export.

In [6]:
df_combined = df_items.merge(
    df_receipts[["receipt_id", "store", "purchase_date", "scan_quality"]],
    on="receipt_id",
    how="left",
)
df_combined

,item_id,receipt_id,file_name,raw_name,normalized_name,quantity,unit,price,matched_product_id,confidence,uncertain,store,purchase_date,scan_quality
0,1827153b-b012-453b-926f-c3df158cae91,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Banane Fair Bio 2,35 A",Banane Fair Bio,1.182,kg,2.35,None,0.5,True,Lidl,2026-06-30,good
1,592f30db-ea4d-4418-a314-b03d96f7a074,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Zwiebeln weiß 1,28 A",Zwiebeln weiß,0.356,kg,1.28,None,0.5,True,Lidl,2026-06-30,good
2,97af5108-17a5-480b-af76-e0f0b2e841f3,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bioland Zucchini 1,49 A",Bioland Zucchini,1.000,piece,1.49,None,0.5,True,Lidl,2026-06-30,good
3,c3138c2f-675b-4c0d-9b6b-37c3472fcd15,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Petersilie 1,39 A",Petersilie,1.000,piece,1.39,None,0.5,True,Lidl,2026-06-30,good
4,517fc874-ad36-4723-9408-0ba10f7861d5,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Limetten 1,99 A",Bio Limetten,1.000,piece,1.99,None,0.5,True,Lidl,2026-06-30,good
5,c9574418-4e27-40cd-b21c-4fab8a121b8b,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Weidemilch 3,8% 1,19 A",Weidemilch,1.000,piece,1.19,None,0.5,True,Lidl,2026-06-30,good
6,5cc0d5e7-9b8b-483e-a5c4-0159e175d6a9,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Exquisa fitlineNatur 1,59 A",Exquisa fitlineNatur,1.000,piece,1.59,None,0.5,True,Lidl,2026-06-30,good
7,6bf23279-4d79-4a28-a525-71f9f4ee9f2c,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio-Eier OKT 10er 3,99 A",Bio-Eier OKT 10er,1.000,piece,3.99,None,0.5,True,Lidl,2026-06-30,good
8,1edac537-14b2-4b64-a470-76646701ce9e,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Müsli Schoko Kn. 2,65 A",Bio Müsli Schoko Kn,1.000,piece,2.65,None,0.5,True,Lidl,2026-06-30,good
9,cd8d9096-6e3c-420a-b96e-c2e89daa45c6,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Lava Bar Cookie 1,69 A",Lava Bar Cookie,1.000,piece,1.69,None,0.5,True,Lidl,2026-06-30,good


## Export (optional)

Writes the two tables to disk next to this notebook, for downstream analysis without re-running OCR.

In [7]:
df_receipts.to_csv(REPO_ROOT / "receipt_extraction_receipts.csv", index=False)
df_items.to_csv(REPO_ROOT / "receipt_extraction_items.csv", index=False)
df_combined.to_csv(REPO_ROOT / "receipt_extraction_combined.csv", index=False)
print("Saved:")
print(" -", REPO_ROOT / "receipt_extraction_receipts.csv")
print(" -", REPO_ROOT / "receipt_extraction_items.csv")
print(" -", REPO_ROOT / "receipt_extraction_combined.csv")

Saved:
 - /Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipt_extraction_receipts.csv
 - /Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipt_extraction_items.csv
 - /Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipt_extraction_combined.csv


## Matching preview: BLS / OFF / fallback category

For inspection only. For each parsed item's `normalized_name`, this calls the real matching services directly (no reimplementation):

- `services/bls_matcher.match_product_bls` — the BLS match the tiered resolver would pick, plus `services/bls_matcher.search_bls` — the raw top-5 closest BLS candidates by name similarity (independent of whether any clears the match threshold).
- `services/matcher.match_product` — the OpenFoodFacts match the tiered resolver would pick, plus `services/off_api.search_products` — the raw top-5 closest OFF candidates.
- `services/fallback_categories._canonical_category` — the category a rough nutrition estimate would use if neither BLS nor OFF produced a usable match (same call the offline parser itself makes, `_canonical_category(None, name)`).

This makes real network calls to the public OpenFoodFacts search API (cached on disk in `_off_cache.json` after the first call, same as the running app) and scans the in-memory BLS reference table already loaded by `bls_matcher.py`. Nothing is written to Supabase.

In [8]:
from backend.app.services import bls_matcher, off_api, fallback_categories
from backend.app.services.matcher import match_product
from backend.app.services.text_similarity import token_similarity


def _off_match_str(name: str):
    m = match_product(name)
    if m is None:
        return None
    return f"{m.matched_name} [off_id={m.off_id}] conf={m.confidence:.2f} ({m.match_type.value})"


def _off_top5_str(name: str, page_size: int = 5):
    candidates = off_api.search_products(name, page_size=page_size)
    scored = []
    for p in candidates:
        display = off_api.product_display_name(p) or "(no name)"
        scored.append(f"{display} ({token_similarity(name, display):.2f})")
    return " | ".join(scored) if scored else None


def _bls_match_str(name: str):
    m = bls_matcher.match_product_bls(name)
    if m is None:
        return None
    return f"{m['matched_name']} [bls_code={m['bls_code']}] conf={m['confidence']:.2f} ({m['match_type']})"


def _bls_top5_str(name: str, page_size: int = 5):
    candidates = bls_matcher.search_bls(name, page_size=page_size)
    return " | ".join(f"{c['name_de']} ({c['score']:.2f})" for c in candidates) if candidates else None


bls_match, bls_top5, off_match, off_top5, fallback_cat = [], [], [], [], []
query_names = df_items["normalized_name"].fillna(df_items["raw_name"])

for i, name in enumerate(query_names):
    bls_match.append(_bls_match_str(name))
    bls_top5.append(_bls_top5_str(name))
    off_match.append(_off_match_str(name))
    off_top5.append(_off_top5_str(name))
    fallback_cat.append(fallback_categories._canonical_category(None, name))
    if (i + 1) % 20 == 0 or (i + 1) == len(query_names):
        print(f"matched {i + 1}/{len(query_names)} items")

df_items["BLS_match"] = bls_match
df_items["closest_5_after_BLS"] = bls_top5
df_items["OFF_match"] = off_match
df_items["closest_5_after_OFF"] = off_top5
df_items["Fallback_category"] = fallback_cat

df_items

matched 20/119 items


matched 40/119 items


matched 60/119 items


matched 80/119 items


matched 100/119 items


matched 119/119 items


,item_id,receipt_id,file_name,raw_name,normalized_name,quantity,unit,price,matched_product_id,confidence,uncertain,BLS_match,closest_5_after_BLS,OFF_match,closest_5_after_OFF,Fallback_category
0,1827153b-b012-453b-926f-c3df158cae91,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Banane Fair Bio 2,35 A",Banane Fair Bio,1.182,kg,2.35,None,0.5,True,Banane roh [bls_code=F503100] conf=0.85 (fuzzy),"Banane roh (0.85) | Bananenchips frittiert, gesüßt (0.85) | Kochbanane roh (...",Banane BIO [off_id=3250393062582] conf=1.00 (fuzzy),Barres céréales Banane Chocolat noir (0.85) | Banane BIO (1.00) | (no name) ...,fruit
1,592f30db-ea4d-4418-a314-b03d96f7a074,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Zwiebeln weiß 1,28 A",Zwiebeln weiß,0.356,kg,1.28,None,0.5,True,"Weiße Rübe/Wasserrübe/Herbstrübe, roh [bls_code=G614100] conf=0.85 (fuzzy)","Weiße Rübe/Wasserrübe/Herbstrübe, roh (0.85) | Heilbutt/Weißer Heilbutt, roh...",Zwiebeln weiss [off_id=4063367118326] conf=0.89 (fuzzy),Zwiebeln weiss (0.89) | Gemüse - Zwiebeln weiss (0.85),vegetable
2,97af5108-17a5-480b-af76-e0f0b2e841f3,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bioland Zucchini 1,49 A",Bioland Zucchini,1.000,piece,1.49,None,0.5,True,"Schwein Speck, ohne Schwartenzug (S VII) roh [bls_code=U605700] conf=0.85 (f...","Schwein Speck, ohne Schwartenzug (S VII) roh (0.85) | Eierteigwaren Tortello...",NaN,Zucchini (0.85),other
3,c3138c2f-675b-4c0d-9b6b-37c3472fcd15,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Petersilie 1,39 A",Petersilie,1.000,piece,1.39,None,0.5,True,Petersilienblatt getrocknet [bls_code=G250400] conf=0.85 (fuzzy),Petersilienblatt getrocknet (0.85) | Petersilienblatt roh (0.85) | Wurzelpet...,Petersilie getrocknet [off_id=20226558] conf=0.85 (fuzzy),Pesto Basilico e Rucula (0.48) | Petersilie getrocknet (0.85) | Petersilie -...,other
4,517fc874-ad36-4723-9408-0ba10f7861d5,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Limetten 1,99 A",Bio Limetten,1.000,piece,1.99,None,0.5,True,Limette roh [bls_code=F602100] conf=0.85 (fuzzy),Limette roh (0.85) | Limettensaft (0.85) | Limettensaft Konzentrat (0.85) | ...,Limetten Saft [off_id=4003387006012] conf=0.85 (fuzzy),Limetten Saft (0.85) | Bio Limetten (1.00) | Bio-Fairtrade-Limetten (1.00) |...,other
5,c9574418-4e27-40cd-b21c-4fab8a121b8b,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Weidemilch 3,8% 1,19 A",Weidemilch,1.000,piece,1.19,None,0.5,True,"Milch fettarm, frisch, 1,5 % Fett, pasteurisiert [bls_code=M111200] conf=0.8...","Milch fettarm, frisch, 1,5 % Fett, pasteurisiert (0.85) | H-Milch fettarm, 1...",Weidemilch [off_id=4061462544262] conf=1.00 (exact),Frische Weidemilch (0.85) | frische Weidemilch (0.85) | Deutsche Markenbutte...,dairy
6,5cc0d5e7-9b8b-483e-a5c4-0159e175d6a9,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Exquisa fitlineNatur 1,59 A",Exquisa fitlineNatur,1.000,piece,1.59,None,0.5,True,"Cerealien aus Vollkornweizenschrot, gesüßt, ungefüllt, Natur/Frucht/Schokola...","Cerealien aus Vollkornweizenschrot, gesüßt, ungefüllt, Natur/Frucht/Schokola...",NaN,NaN,other
7,6bf23279-4d79-4a28-a525-71f9f4ee9f2c,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio-Eier OKT 10er 3,99 A",Bio-Eier OKT 10er,1.000,piece,3.99,None,0.5,True,"Rhabarberkompott mit einer Zuckerart und Süßungsmitteln, Konserve [bls_code=...","Rhabarberkompott mit einer Zuckerart und Süßungsmitteln, Konserve (0.89) | P...",NaN,NaN,protein
8,1edac537-14b2-4b64-a470-76646701ce9e,1b66426a-f508-4e82-8c63-dae566dd3202,2026.06.30_23001961862026063034796.jpg.png,"Bio Müsli Schoko Kn. 2,65 A",Bio Müsli Schoko Kn,1.000,piece,2.65,None,0.5,True,"Cerealien extrudiert, gesüßt, ungefüllt, mit Frucht-/Honig-/Karamell-/Schoko...","Cerealien extrudiert, gesüßt, ungefüllt, mit Frucht-/H

### Re-export with matching-preview columns

Overwrites the two item-level CSVs from the extraction step above with the same rows plus the five new columns; `receipt_extraction_receipts.csv` is untouched.

In [9]:
df_combined = df_items.merge(
    df_receipts[["receipt_id", "store", "purchase_date", "scan_quality"]],
    on="receipt_id",
    how="left",
    suffixes=("", "_receipt"),
)

df_items.to_csv(REPO_ROOT / "receipt_extraction_items.csv", index=False)
df_combined.to_csv(REPO_ROOT / "receipt_extraction_combined.csv", index=False)
print("Re-saved with matching-preview columns:")
print(" -", REPO_ROOT / "receipt_extraction_items.csv")
print(" -", REPO_ROOT / "receipt_extraction_combined.csv")

Re-saved with matching-preview columns:
 - /Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipt_extraction_items.csv
 - /Users/stuartkasemeier/Documents/Work/07-Arbeitslosigkeit/ai_project_management/neuefische_course/AIPM_capstone_project_supermarket_optimizer/receipt_extraction_combined.csv
